In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to gather information about Lunapolis, including its capital, weather, population of cheese miners, and the current labor situation affecting the cheese miners' union.\n\n## SUMMARY\n- The capital of the moon is identified as Lunapolis.\n- The weather in Lunapolis is reported as clear skies, with a high temperature of 120°C and a low of -100°C.\n- The population of cheese miners living in Lunapolis is stated to be 100,000.\n- It is suggested that the cheese miners' union is likely to strike due to discontent with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='30d07501-2acc-4222-9648-dc3ab708ec68'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to gather information about Lunapolis, including its capital, weather, population of cheese miners, and the current labor situation affecting the cheese miners' union.

## SUMMARY
- The capital of the moon is identified as Lunapolis.
- The weather in Lunapolis is reported as clear skies, with a high temperature of 120°C and a low of -100°C.
- The population of cheese miners living in Lunapolis is stated to be 100,000.
- It is suggested that the cheese miners' union is likely to strike due to discontent with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='92796ff7-1202-4756-95be-d215355503c3'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='5cb4cff0-834e-4cad-b2c2-141f16b6c41e', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='5cfb4b42-f2b2-42b6-8d13-c2b103b3c289'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='2cf20d61-7123-4392-8ddc-eb62cba725a4', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='3df12e34-91f3-4bbe-8ed8-faeadafc1bc9'),
              AIMessage(content='I can’t read the device’s temperature from here, and since it 

In [8]:
print(response["messages"][-1].content)

I can’t read the device’s temperature from here, and since it won’t turn on you can’t check temps from the software. But you can check for overheating and proceed with safety-focused troubleshooting.

What to do now
- Safety first: if the case feels extremely hot or you smell burning, unplug it immediately and don’t power it on again until it’s cooled.
- Let it cool: after it’s cooled for 15–30 minutes, try to power it on again.
- Check the power situation:
  - Try a different outlet to rule out the wall socket.
  - Inspect the charger/adapter for damage and test with a known-good charger if you have one.
  - If it’s a laptop with a removable battery, remove the battery and try powering with the charger only.
  - Look for any LEDs on the device or the charger when you plug in. A steady or blinking light can indicate a specific issue.
- Listen and observe:
  - Do you hear fans spinning, beeps, or any sounds when trying to power on? Any blinking patterns of lights?
  - Any smell coming f